In [5]:
!pip install pyspark

In [6]:
!wget -O owid-covid-data.csv https://raw.githubusercontent.com/owid/covid-19-data/master/public/data/owid-covid-data.csv

--2026-05-20 09:05:06--  https://raw.githubusercontent.com/owid/covid-19-data/master/public/data/owid-covid-data.csv
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 98391483 (94M) [text/plain]
Saving to: ‘owid-covid-data.csv’

owid-covid-data.csv 100%[===================>]  93.83M   341MB/s    in 0.3s    

2026-05-20 09:05:08 (341 MB/s) - ‘owid-covid-data.csv’ saved [98391483/98391483]



In [8]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, sum, desc

In [9]:
spark = SparkSession.builder \
    .appName("COVID19 Data Analysis") \
    .getOrCreate()

In [16]:
df = spark.read.csv("owid-covid-data.csv", header=True, inferSchema=True)

df.show(5)
df.printSchema()

+--------+---------+-----------+----------+-----------+---------+------------------+------------+----------+-------------------+-----------------------+---------------------+------------------------------+------------------------+----------------------+-------------------------------+-----------------+------------+------------------------+-------------+-------------------------+---------------------+---------------------------------+----------------------+----------------------------------+-----------+---------+------------------------+----------------------+------------------+-------------------------------+-------------+--------------+-----------+------------------+-----------------+-----------------------+--------------+----------------+-------------------------+------------------------------+-----------------------------+-----------------------------------+--------------------------+-------------------------------------+------------------------------+-------------------------------

In [17]:
df.select("location", "date", "total_cases", "total_deaths").show(20)

+-----------+----------+-----------+------------+
|   location|      date|total_cases|total_deaths|
+-----------+----------+-----------+------------+
|Afghanistan|2020-01-05|          0|           0|
|Afghanistan|2020-01-06|          0|           0|
|Afghanistan|2020-01-07|          0|           0|
|Afghanistan|2020-01-08|          0|           0|
|Afghanistan|2020-01-09|          0|           0|
|Afghanistan|2020-01-10|          0|           0|
|Afghanistan|2020-01-11|          0|           0|
|Afghanistan|2020-01-12|          0|           0|
|Afghanistan|2020-01-13|          0|           0|
|Afghanistan|2020-01-14|          0|           0|
|Afghanistan|2020-01-15|          0|           0|
|Afghanistan|2020-01-16|          0|           0|
|Afghanistan|2020-01-17|          0|           0|
|Afghanistan|2020-01-18|          0|           0|
|Afghanistan|2020-01-19|          0|           0|
|Afghanistan|2020-01-20|          0|           0|
|Afghanistan|2020-01-21|          0|           0|


In [18]:
df_valid = df.filter(col("total_cases") > 0)

In [19]:
df_clean = df_valid.select(
    "location",
    "date",
    "total_cases",
    "total_deaths"
).dropna()

In [20]:
pakistan_data = df_clean.filter(col("location") == "Pakistan")
pakistan_data.show()

+--------+----------+-----------+------------+
|location|      date|total_cases|total_deaths|
+--------+----------+-----------+------------+
|Pakistan|2020-03-01|          2|           0|
|Pakistan|2020-03-02|          2|           0|
|Pakistan|2020-03-03|          2|           0|
|Pakistan|2020-03-04|          2|           0|
|Pakistan|2020-03-05|          2|           0|
|Pakistan|2020-03-06|          2|           0|
|Pakistan|2020-03-07|          2|           0|
|Pakistan|2020-03-08|          6|           0|
|Pakistan|2020-03-09|          6|           0|
|Pakistan|2020-03-10|          6|           0|
|Pakistan|2020-03-11|          6|           0|
|Pakistan|2020-03-12|          6|           0|
|Pakistan|2020-03-13|          6|           0|
|Pakistan|2020-03-14|          6|           0|
|Pakistan|2020-03-15|         21|           0|
|Pakistan|2020-03-16|         21|           0|
|Pakistan|2020-03-17|         21|           0|
|Pakistan|2020-03-18|         21|           0|
|Pakistan|202

In [21]:
df_clean.select(
    sum("total_cases").alias("Total Cases"),
    sum("total_deaths").alias("Total Deaths")
).show()

+-------------+------------+
|  Total Cases|Total Deaths|
+-------------+------------+
|3033056852746| 33463017649|
+-------------+------------+



In [22]:
top_countries = df_clean.groupBy("location") \
    .sum("total_cases") \
    .orderBy(desc("sum(total_cases)"))

top_countries.show(10)

+--------------------+----------------+
|            location|sum(total_cases)|
+--------------------+----------------+
|               World|    715697182101|
|High-income count...|    393562796886|
|                Asia|    252167317226|
|              Europe|    236756684151|
|Upper-middle-inco...|    218037426784|
| European Union (27)|    171330559623|
|       North America|    127073670231|
|       United States|    105914483457|
|Lower-middle-inco...|    100873065026|
|       South America|     73484570403|
+--------------------+----------------+
only showing top 10 rows


In [23]:
spark.stop()